In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[3, 5, 7, 11, 13, 15, 18, 20, 23, 25, 27, 28, 31])

nb_name = "PILOT_trajectory.ipynb"

_compare_version(nb_name)

# Trajectory analysis with PILOT
<hr style="border:2px solid black"> </hr>

## 1 -  Description

**Welcome to our extented notebook for scRNA/pathomics Data for unsupervised trajectory analysis with PILOT:**

PILOT is a python library developed by Costalab to perform patient-level analysis by using optimal transport. The objective is to enable the direct comparison between multiple scales samples e.g. control vs. disease and the observation of gene expression at different timepoints.
    
    
**Reference**: Joodaki et al. (2024) Detection of PatIent-Level distances from single cell genomics and pathomics data with   
    Optimal Transport (PILOT), Molecular Systems Biology, Volume 20, https://pubmed.ncbi.nlm.nih.gov/38177382/ 

<div class="alert alert-block alert-danger">
    
**Requires a clustered or otherwise categorized anndata object. A clustering can be generated with a clustering notebook (e.g. <code>rna_analysis/notebooks/04_clustering.ipynb</code>).**
    
**Move this notebook into the notebook folder (e.g. <code>rna_analysis/notebooks/</code>) of the respective analysis before using it!**

</div>

-----------

## 2 - Setup

### 2.1 Loading packages

In [ ]:
from sctoolbox import settings
import scanpy as sc
import pandas as pd
import pilotpy as pl
import sctoolbox
import sctoolbox.utils as utils

-----------

## 3 - Load anndata

### 3.1 Loading anndata

<div class="alert alert-block alert-danger">
    
**If your anndata is already clustered and was not generated with the sc_framework, it is necessary to utilize the Assembly Notebook. In the absence of such a file, the Anndata must be generated with Notebooks 1-4.**
    
</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
clustered_adata = "anndata_5.h5ad"

In [ ]:
adata = utils.adata.load_h5ad(clustered_adata) 

with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata)
    display(adata.obs)
    display(adata.var)
    display(adata.obsm)

### 3.2 Impute missing values in the data using MAGIC algorithm

More frequent undersampling or simply missing values of mRNA molecules can lead to the loss of important gene-gene relationships or, in the case of PILOT, to incorrect calculations when it comes to e.g. the Fold change. To avoid this, the MAGIC (Markov Affinity-based Graph Imputation of Cells) algorithm is used to denoise the cell count matrix and supplement missing transcripts through data diffusion for all genes across cells.

The MAGIC algorithm is based on Markov affinity-based graph imputation, which uses the Markov model to estimate the probability of a transcript being present in a cell.

Reference: Van Dijk et al. Recovering gene interactions from single-cell data using data diffusion. Cell, 174(3):716–729.e27, jul 2018. https://www.sciencedirect.com/science/article/pii/S0092867418307244?via%3Dihub

**Parameters**

* **name_list**: Denoised genes to return. The default 'all_genes'/None may require a large amount of memory if the input data is sparse. Another possibility is 'pca_only'.

* **t**: Power to which the diffusion operator is powered. This sets the level of diffusion. If ‘auto’, t is selected according to the Procrustes disparity of the diffused data.

* **solver**: Which solver to use. “exact” uses the implementation described in van Dijk et al. [2018]. “approximate” uses a faster implementation that performs imputation in the PCA space and then projects back to the gene space. Note, the “approximate” solver may return negative values.





<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Denoised genes to return
name_list = 'all_genes'
    
# time-steps to use for diffusion 
t = 3

# Which solver to use
solver =  'exact'

In [ ]:
sc.external.pp.magic(adata, 
                     name_list=name_list, 
                     t=t, 
                     solver=solver)

-----------

## 4 - General input

### 4.1 Input

<div class="alert alert-block alert-danger">

**The coulumn for the <code>sample</code> parameter needs at least five different categories to calculate the diffusionmap.**

</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Provide the name of the variable in the obsm level of your Anndata that holds the dimension reduction (PCA representation).
emb_matrix = 'X_pca'

# Specify the name of the column in the obs level that corresponds to cell types or clusters.
clusters = 'celltypes'

# Indicate the column name in the obs level that contains information about samples or patients.
sample  = 'Sample'

# Provide the column name in the obs level that represents the status or disease (e.g., “control” or “case”).
status = 'timepoint'

# Add the celltypes to remove 
removing_c = ['Blood - Immune cells', 'Blood - red blood cells']

#Checking the required number of samples 
if len(adata.obs[sample].cat.categories) < 5: 
    print("Unfortunately, it is not possible to perform the analysis with this particular dataset. For pseudotime analysis, please consider the Palantir or pseudotime_analysis Notebook.")

### 4.2 Remove specific celltypes/cluster 

In special cases, it may be necessary to remove certain cell types/clusters (e.g., in the case of contamination by blood cells) or they may simply not be of interest. In these cases, you can use the cell below to remove them from the anndata.


In [ ]:
for celltype in removing_c: 
    adata = adata[adata.obs[clusters] != celltype]

-----------

## 5 - Computing Wasserstein distance

To find similarities in celltypes and samples, this module uses optimal transport according to Pyré & Cuturi (Reference:https://arxiv.org/abs/1803.00567). For that, each sample/patient is modeled as a distribution of cells in clusters (either a specific celltype or -state). The optimal transport is then used to create the transport plan T, assuming a mass movement between cluster pairs.
    
To explain it more clearly, here is a example of myocardial infarction which Costalab used as a reference: 
 * some cellular changes e.g. conversion of healthy cardiomyocyles into damage cardiomyocyles material represents short-    term events in the disease progression --> low costs 
 * on the other hand tissue remodeling e.g. scar formation process represent long term events --> high costs 

</div>

### 5.1 Input

In [ ]:
pl.tl.wasserstein_distance(adata,
                           emb_matrix=emb_matrix,
                           clusters_col=clusters,
                           sample_col=sample,
                           status=status)

### 5.2 Ploting the Cost matrix and the Wasserstein distance heatmaps:

After the computation of the optimal transport plan, we get the heatmaps of the cost matrix and the wasserstein distance. The transport between similar cell clusters represents a low cost (darker blue) and thus transport between more different cell clusters has higher cost (lighter blue). The similarities in total are represented with a dendrogram.
    
* **Costmatrix:** Here you can see the similarities between celltypes/clusters. 
    
* **Wasserstein distance:** This plot shows the similarities between the timepoints/samples. It is estimated as the total cost of the optimal transport. 

In [ ]:
pl.pl.heatmaps(adata, 
               path="../figures/pilot/")

-----------

## 6. Plotting Diffusion map of Wasserstein distance and estimate the trajectory 

### 6.1 Diffusion map of Wasserstein distance 

From the Wasserstein distance, we obtain the distance matrix **W**, which specifies the distance between all samples and is used for calculating of the diffusion map, followed by a path inference algorithm for determining the course of the disease or development. 

A diffusion map is used to achieve dimensionality reduction or feature extraction by calculating a series of embeddings of data using eigenvectors and eigenvalues to obtain the coordinates. Dimension reduction takes two coordinates from the 3D map to represent the shape, in our case the trajectory, in a 2D map, the diffusion map. 

The visualization displays the trajectory. It takes the categories from the parameter <code>status</code>. 

In [ ]:
# Set the plot title 
plot_title = "zebrafish heart development"

In [ ]:
pl.pl.trajectory(adata, 
                 path="../figures/pilot/",  
                 plot_title=plot_title)

### 6.2 Estimate the pseudotime

<div class="alert alert-block alert-danger">

**which is used for the following plots! For that, it is important to define the start of the trajectory with the help of the parameter <code>source_node</code>.** 

To do this, proceed as follows:
As you can see after starting the cell below, this function takes the diffusionmap and compute a principal graph with nodes over it (The red points are the points of the diffusionmap). After that, if you have timepoints look at the diffusionmap and find the starting point. Now look at the nodes and find the node near this point and set it as the <code>source_node</code>. In case of patients samples look where you can find the control/healthy samples, take the node at the end of the graph and set this one as the <code>source_node</code>.
    
**Pay attention to restart this function and all following ones if you changed the <code>source_node</code>!**

</div>

**Parameters:**
    
*   **<code> NumNodes</code>**: The number of nodes within the principal graph. The higher the number the more the nodes nestle on the timepoints, which may result in a better estimation.
    
*  **<code>source_node</code>**: The first timepoint or the patient/sample which determine the start of the trajectory.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
NumNodes = 20
source_node = 2

In [ ]:
pl.pl.fit_pricipla_graph(adata, 
                         NumNodes=NumNodes, 
                         source_node=source_node, 
                         path="../figures/pilot/")

-----------

## 7 - Cell proportion

Next, we get the cell proportion for each time point using a robust regression model to find groups of cells where the number of cells correlates with the pseudotime progression. While the heatmap reveals the cell proportion with the sorted pseudotime on the right, the plot below visualizes it in total with the regression model.

**Description for the second plot:**
* X-axis: Cell coverage
* Y-axis: Samples = pseudotime
* Points: Samples
* Line: median trend 

<h1><center>⬐ Fill in input data here ⬎</center></h1>

**Before starting this function, please decide how you want to label the x-axis by determining if you have timepoints or samples with e.g. patient id.**

* **"timepoints"**: Labels the x-axis for a better overview. 
* **"samples"**: In case of samples where it might be unnecessary to show the label, the aixs simply displays integers instead.

In [ ]:
# Determine if you have "timepoints" or "samples" 
axis = "timepoints"

In [ ]:
pl.tl.cell_importance(adata, 
                      axis=axis, 
                      path_plt="../figures/pilot/", 
                      path_table="../tables/pilot/")

-----------

## 8. Finding gene markers for the clusters

Now that we have gained initial insight into the progression of the celltypes within the trajectory using the estimate pseudotime, we would like to examine the genes associated with this trajectory for each celltype, whose expression changes continuously and significantly with progression.

Three regression models are used for this purpose, with the most significant model being specified for each gene: 

+ linear
+ quadratic
+ quadratic-linear 
 

After running the command, you can find a folder named ‘Markers’ under "../tables/pilot/". There, we will have a folder for each cell type. The file ‘Whole_expressions.csv’ contains all statistics associated with genes for that cell type.

<div class="alert alert-block alert-danger">
    
**Note: This is resource-intensive. May take >1hr per celltype/cluster.**
    
</div>

In [ ]:
for cell in adata.uns['cellnames']:
    pl.tl.genes_importance(adata,
                           name_cell=cell,
                           sample_col=sample,
                           col_cell=clusters,
                           plot_genes=False,
                           path="../tables/pilot/")

-----------

## 9. Group genes markers by pattern


Here, we cluster genes based on the pattern found for each and plot their heatmap. Below the heatmap, we depict the pattern of each group's 
genes and top 10 genes having significant changes through disease progression.

You can find curves activities' statistical scores that show the fold changes of the genes through disease progression in the Markers folder for each cell type separately.

#### Plot descriptions 

1. Heatmap: The heatmap represents the expression of genes 

2. Expressionpattern per cluster: 

3. Top genes per cluster: 

4. Expressionpatterns of the top genes per cluster:

5. GO enrichment per cluster: It may be that for the set of genes of the cluster the genes. 



<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Set the range of the cluster
scaler_value = 0.4

In [ ]:
pl.pl.genes_selection_analysis(adata, 
                               cluster, 
                               scaler_value=scaler_value, 
                               organism=organism, 
                               path_to_tables="../tables/pilot/", 
                               path_to_figures="../figures/pilot/")

-----------

## 10. Hallmarker genes 

Here, we utilize the [Enrichr](https://maayanlab.cloud/Enrichr/) tool to identify the hallmarks and to collect enriched GO pathways for the clustered genes. Hallmarks gene sets summarizes the relevant pathway informations from multiple "founder" sets of the database. As a result of that the gene sets are refined and concise by reducing variation and redundancy. For this "The Molecular Signatures Database" (MSigDB) is used for the organism human and mouse, which provides various gen sets of metabolic diseases and biologic processes.

**Enrichr libraries for human:**
https://maayanlab.cloud/Enrichr/#libraries

### 10.1 Select organism and library

<div class="alert alert-block alert-danger">

**Please clarify your organism of interest by using the parameter <code>organism</code>. Since Enrichr is used for this function, the possible organisms are unfortunately limited.
You must consult the website below to find the suitable library for the desired organism, which you pass to the parameter <code>dataset</code>.**
    
Organism and their ID: 

| Organism  | Enrichr-ID | Recommended library |
|--------|-------------|-------------|
|*H. sapiens*|"human"| "MSigDB_Hallmark_2020"|
|*M. musculus*|"mouse"| "MSigDB_Hallmark_2020"|
|*D. rerio*|"fish"| "GO_Biological_Process_2018"|
|*C. elegans*|"worm"| "GO_Biological_Process_2018"|
|*S. cerevisiae*|"yeast"| "GO_Biological_Process_2018"|
|*D. melanogaster*|"fly"|  "GO_Biological_Process_2018"|   
    
**libraries overview for selected organism:**
https://maayanlab.cloud/modEnrichr/

</div>

### Explanation for some Library terms

**Molecular Function**  
Molecular-level activities performed by gene products. Molecular function terms describe activities that occur at the molecular level, such as “catalysis” or “transport”. GO molecular function terms represent activities rather than the entities (molecules or complexes) that perform the actions, and do not specify where, when, or in what context the action takes place. Molecular functions generally correspond to activities that can be performed by individual gene products (i.e. a protein or RNA), but some activities are performed by molecular complexes composed of multiple gene products. Examples of broad functional terms are catalytic activity and transporter activity; examples of narrower functional terms are adenylate cyclase activity or Toll-like receptor binding. To avoid confusion between gene product names and their molecular functions, GO molecular functions are often appended with the word “activity” (a protein kinase would have the GO molecular function protein kinase activity). 

+ Examples: 
  + GO_Molecular_Function
  + KEGG
  
  
**Cellular Component**  
A location, relative to cellular compartments and structures, occupied by a macromolecular machine. There are two ways in which the gene ontology describes locations of gene products: (1) the cellular anatomical entities, in which a gene product carries out a molecular function. Cellular anatomical entities includes cellular structures such as the plasma membrane and the cytoskeleton, as well as membrane-enclosed cellular compartments such as the mitochondrion, and (2) the stable macromolecular complexes of which they are parts, e.g., the clathrin complex.  

+ Examples: 
  + GO_Cellular_Component
  + Anatomy_AutoRIF

**Biological Process**  
The larger processes, or ‘biological programs’ accomplished by multiple molecular activities. Examples of broad biological process terms are DNA repair or signal transduction. Examples of more specific terms are pyrimidine nucleobase biosynthetic process or glucose transmembrane transport. Note that a biological process is not equivalent to a pathway. At present, the GO does not try to represent the dynamics or dependencies that would be required to fully describe a pathway.

+ Examples: 
  + GO_Biological_Process
  + WikiPathways

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Specific the dataset of interest
dataset = 'GO_Cellular_Component_GeneRIF'

# Specific the organism of interest
organism = "fish"

### 10.2 Clustering of genes and identification of  pathways

This function generates the first two plots of the previous genes_selection_analysis function. The 

In [ ]:
pl.pl.genes_selection_heatmap(adata, 
                              cluster, 
                              scaler_value=scaler_value, 
                              path_to_results="../tables/pilot/")

The hallmark-plot shows a heatmap with the clusters on the x-axis and the GO paths of the selected library on the right y-axis.

In [ ]:
dic = {"human": "hsapiens", "fish": "drerio", "mouse": "mmusculus", "worm": "celegans",
        "yeast": "scerevisiae", "fly": "dmelanogaster"}

if organism in dic:
    pl.pl.plot_hallmark_genes_clusters(adata, 
                                       cluster, dataset, 
                                       path="../figures/pilot/", 
                                       organism=id_organism)
else:
    raise Exception("No analysis possible with your organism of interest")

-----------

## 11 Finding specifc marker for given cluster 

The previous test only finds genes with significant changes over time for a given cell type. However, it does not consider if a similar pattern and expression values are found in other clusters. To further select genes, we use a Wald test that compares the fit of the gene in the cluster vs. the fit of the gene in other clusters.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Number of top genes to consider for each expression pattern
number_genes = 40 

# clusters/celltypes of interest
cellnames = ['Myocardium', "Endocardium", "Epicardium", "Fibroblasts "]

<div class="alert alert-block alert-danger">
    
**Note: This is resource-intensive. May take >1hr per celltype/cluster.**
    
</div>

In [ ]:
pl.tl.gene_cluster_differentiation(adata,
                                   cellnames=cellnames, 
                                   number_genes=number_genes, 
                                   path="../tables/pilot/")

### 11.1 Further filtering based on FC and p-value

To find a final list of genes, we only consider genes with a fold change higher than 0.5, i.e. genes which expression is increased in the cluster at hand and we sort the genes based on the Wald test p-value. These can be seen below. 

The filtered table is saved as "gene_clusters_stats_extend_[celltype]_filtered.csv".

<div class="alert alert-block alert-danger">
    
**Please set from here the cluster/celltype of interest you want to have a more further analysis in the following functions.**
    
</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
#The celltype/cluster of interest 
cluster = 'Endocardium'

#The number of rows of the table to be displayed
head_number = 100

# Set the thershold of fold change 
fc = 0.5

# Set the thershold of p-value
pvalue = 0.1

In [ ]:
df = pl.tl.results_gene_cluster_differentiation(cluster_name=cluster,
                                                path="../tables/pilot/",
                                                threshold=fc,
                                                p_value=pvalue)
                                
with pd.option_context("display.max.rows", None, "display.max.columns", None):
     display(df.head(head_number))

-----------

## 12. Expression pattern of specfic genes

### 12.1 Expression pattern over time 

We can visualize the expression pattern of specific genes over time for the given cluster. Again on the x-axis the pseudotime 

<h1><center>⬐ Fill in input data here ⬎</center></h1>

<div class="alert alert-block alert-danger">
    
**Look in previous table for the gene of interest in the column.**
    
</div>

In [ ]:
#The gene of interest 
gene_list = ['ddx39ab','adarb1a','sptlc2b','si:ch1073-291c23.2','CABZ01080568.1']

<div class="alert alert-block alert-danger">
    
**Before starting this function, please decide how you want to label the x-axis by determining if you have timepoints or samples with e.g. patient id.**

* **"timepoints"**: Labels the x-axis for a better overview. 
* **"samples"**: In case of samples where it might be unnecessary to show the label, the aixs simply displays the integers instead.

</div>

In [ ]:
# Determine if you have "timepoints" or "samples" 
axis = "timepoints"

In [ ]:
pl.pl.plot_gene_list_pattern(gene_list, 
                             cluster, 
                             path_to_tables="../tables/pilot/", 
                             axis=axis)

### 12.2 Comparing expression pattern with other clusters

In the plot below, the orange line indicates the fit in the target cell type (shown as orange lines) compared to other cell types (represented by grey lines).

In [ ]:
pl.pl.exploring_specific_genes(adata, cluster_name=cluster,
                               gene_list=gene_list, 
                               path_to_tables="../tables/pilot/",
                               path_to_figures="../figures/pilot/", 
                               axis=axis)

-----------

## 13. Go enrichment for specific marker

Here is the GO enrichment for  the 50 first top genes of the selected celltype. Unlike before [g:Profiler](https://biit.cs.ut.ee/gprofiler/gost) is used instead of Enrichr.  Plot is saved at the folder "GO". describe gprofiler

<div class="alert alert-block alert-danger">

**Note that the previous default organism is used for this function if it was possible. For all others you can use the parameter <code>organism_for_gp</code>. Since gprofiler is used here for the go enrichment, you need to look up on the website below in order to find the ID for your interest organism which you are hand over to the parameter.**

    
Here are the IDs of the organism from the hallmarker analysis: 

| Organism  | gprofil-ID |
|--------|-------------|
|human|"hsapiens"|
|mouse|"mmusculus"|
|rat|"rnorvegicus"|
|*C. elegans*|"celegans"|
|zebrafish|"drerio"| 
|*S. cerevisiae*|"scerevisiae"|   
|fruit fly|"dmelanogaster"| 
    
    
**Organism-ID for gProfiler:**
https://biit.cs.ut.ee/gprofiler/page/organism-list


</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# The organism of interest if it's not listed in the table
organism_for_gp = "drerio"

In [ ]:
if organism in dic:
    id_organism = dic[organism]
    
    pl.pl.go_enrichment(df, 
                        cell_type=cluster, 
                        organism=id_organism, 
                        path_plt="../figures/pilot/", 
                        path_table="../tables/pilot/")
else: 
    pl.pl.go_enrichment(df, 
                        cell_type=cluster, 
                        organism=organism_for_gp, 
                        path_plt="../figures/pilot/", 
                        path_table="../tables/pilot/")

-----------

## 14. Saving adata

In [ ]:
utils.adata.save_h5ad(adata, "anndata_pilot.h5ad")

In [ ]:
settings.close_logfile()